In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot
import seaborn as sns

In [2]:
df = pd.read_csv('train.txt',sep =';',header = None,names = ['text','emotions'])

In [3]:
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

text        0
emotions    0
dtype: int64

In [5]:
unique_emotions = df['emotions'].unique()

In [6]:
emotion_numbers= {}
i = 0
for emo in unique_emotions:
    emotion_numbers[emo]=i
    i+=1
df['emotions']= df['emotions'].map(emotion_numbers)

In [7]:
df

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [8]:
df['text']= df['text'].apply(lambda x: x.lower())

In [9]:
import string
def rem_pun(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [10]:
df['text']= df['text'].apply(rem_pun)

In [11]:
def rem_num(txt):
    new=""
    for i in txt:
        if not i.isdigit():
            new= new+i
    return new

df['text']=df['text'].apply(rem_num)

In [12]:
import nltk

In [13]:
from nltk.corpus import stopwords

In [14]:
from nltk.tokenize import word_tokenize

In [15]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\atish\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\atish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
stop_words = set(stopwords.words('english'))


In [17]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [18]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\atish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [19]:
def remove(txt):
    word = word_tokenize(txt)
    cleaned= []
    for i in word :
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)


    

In [20]:
df['text'] = df['text'].apply(remove)

In [21]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [22]:
df.head()

,text,emotions
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [23]:
from sklearn .model_selection import train_test_split
X_train,X_test, y_train , y_test = train_test_split(df['text'],df['emotions'],test_size=0.20,random_state =42)

In [24]:
X_train

676      refers course though cant help feeling somehow...
12113                im starting feel im suffering fatigue
7077     feel like probably would liked book little bit...
13005                                  really feel awkward
12123    im feeling little grumpy today lame weather te...
                               ...                        
13418    love leave reader feeling confused slightly de...
5390                                         feel delicate
860                          starting feel little stressed
15795             feel stressed tired worn shape neglected
7270         feel someone rude wrongly done something lose
Name: text, Length: 12800, dtype: object

In [25]:
X_test

8756                             ive made week feel beaten
4660                              feel strategy worthwhile
6095                     feel worthless weak say want find
304                                        feel clever nov
8241                      im moved ive feeling kind gloomy
                               ...                        
15578    feel useful pulpit find ironic often question ...
5746             dried bladders ready day im feeling brave
6395                             feel thrilled matter days
7624     woke morning text mr c declaring walking work ...
15245                                            feel dumb
Name: text, Length: 3200, dtype: object

In [26]:
df.shape

(16000, 2)

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


In [28]:
bow_vectorizer = CountVectorizer()

In [29]:
X_train_bow = bow_vectorizer.fit_transform(X_train)

In [30]:
X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116049 stored elements and shape (12800, 13359)>

In [31]:
X_test_bow= bow_vectorizer.transform(X_test)

In [32]:
X_test_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 26934 stored elements and shape (3200, 13359)>

In [33]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [34]:
nb_model = MultinomialNB()

In [35]:
nb_model.fit(X_train_bow,y_train)

MultinomialNB()

In [36]:
pred_bow = nb_model.predict(X_test_bow)

In [37]:
print(accuracy_score(y_test,pred_bow))

0.7678125


In [38]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [39]:
tfidf_vectorizer = TfidfVectorizer()

In [40]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf= tfidf_vectorizer.transform(X_test)

In [41]:
nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [42]:
y_pred = nb2_model.predict(X_test_tfidf)

In [43]:
print(accuracy_score(y_test,y_pred))

0.6609375


In [44]:
from sklearn.linear_model import LogisticRegression

In [45]:
logistic_model = LogisticRegression(max_iter = 1000)

In [46]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [47]:
log_pred = logistic_model.predict(X_test_tfidf)

In [48]:
print(accuracy_score(y_test,log_pred))

0.8615625


In [53]:
import pickle

pickle.dump(nb_model, open("emotion_BowNb_model.pkl", "wb"))
pickle.dump(nb2_model, open("emotion_TfidfNb_model.pkl", "wb"))
pickle.dump(logistic_model, open("emotion_TfidfLog_model.pkl", "wb"))

pickle.dump(bow_vectorizer, open("bow_vectorizer.pkl", "wb"))
pickle.dump(tfidf_vectorizer, open("tfidf_vectorizer.pkl", "wb"))